In [1]:
!pip install -q pandas numpy scikit-learn tqdm

In [2]:
!pip install -q google-genai

In [3]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, mean_absolute_error
from tqdm import tqdm
import json
import re
from google import genai
import os
import random

In [4]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

### Data

In [5]:
from zipfile import ZipFile

with ZipFile('archive.zip', 'r') as zip_ref:
    zip_ref.extractall('yelp_data/')

In [6]:
# load csv
df = pd.read_csv('yelp_data/yelp.csv')

print("Dataset shape:", df.shape)
print("Columns:", df.columns)
df.head(3)

Dataset shape: (10000, 10)
Columns: Index(['business_id', 'date', 'review_id', 'stars', 'text', 'type', 'user_id',
       'cool', 'useful', 'funny'],
      dtype='object')


,business_id,date,review_id,stars,text,type,user_id,cool,useful,funny
0,9yKzy9PApeiPPOUJEtnvkg,2011-01-26,fWKvX83p0-ka4JS3dc6E5A,5,"My wife took me here on my birthday for breakfast and it was excellent. The weather was perfect which made sitting outside overlooking their grounds an absolute pleasure. Our waitress was excellent and our food arrived quickly on the semi-busy Saturday morning. It looked like the place fills up pretty quickly so the earlier you get here the better.\n\nDo yourself a favor and get their Bloody Mary. It was phenomenal and simply the best I've ever had. I'm pretty sure they only use ingredients from their garden and blend them fresh when you order it. It was amazing.\n\nWhile EVERYTHING on the menu looks excellent, I had the white truffle scrambled eggs vegetable skillet and it was tasty and delicious. It came with 2 pieces of their griddled bread with was amazing and it absolutely made the meal complete. It was the best ""toast"" I've ever had.\n\nAnyway, I can't wait to go back!",review,rLtl8ZkDX5vH5nAx9C3q5Q,2,5,0
1,ZRJwVLyzEJq1VAihDhYiow,2011-07-27,IjZ33sJrzXqU-0X6U8NwyA,5,"I have no idea why some people give bad reviews about this place. It goes to show you, you can please everyone. They are probably griping about something that their own fault...there are many people like that.\n\nIn any case, my friend and I arrived at about 5:50 PM this past Sunday. It was pretty crowded, more than I thought for a Sunday evening and thought we would have to wait forever to get a seat but they said we'll be seated when the girl comes back from seating someone else. We were seated at 5:52 and the waiter came and got our drink orders. Everyone was very pleasant from the host that seated us to the waiter to the server. The prices were very good as well. We placed our orders once we decided what we wanted at 6:02. We shared the baked spaghetti calzone and the small ""Here's The Beef"" pizza so we can both try them. The calzone was huge and we got the smallest one (personal) and got the small 11"" pizza. Both were awesome! My friend liked the pizza better and I liked the calzone better. The calzone does have a sweetish sauce but that's how I like my sauce!\n\nWe had to box part of the pizza to take it home and we were out the door by 6:42. So, everything was great and not like these bad reviewers. That goes to show you that you have to try these things yourself because all these bad reviewers have some serious issues.",review,0a2KyEL0d3Yb1V6aivbIuQ,0,0,0
2,6oRAC4uyJCsJl1X0WZpVSA,2012-06-14,IESLBzqUCLdSzSqm0eCSxQ,4,love the gyro plate. Rice is so good and I also dig their candy selection :),review,0hT2KtfLiobPvh6cDC8JQg,0,1,0


In [7]:
df.isna().sum()

,0
business_id,0
date,0
review_id,0
stars,0
text,0
type,0
user_id,0
cool,0
useful,0
funny,0


In [8]:
rating_counts = df['stars'].value_counts().sort_index()
print("Rating distribution:\n", rating_counts)

Rating distribution:
 stars
1     749
2     927
3    1461
4    3526
5    3337
Name: count, dtype: int64


Imbalanced towards 4-5 ratings

In [9]:
df_sample = df[['text', 'stars']].sample(200, random_state=42).reset_index(drop=True)
# print(df_sample)

Intentionally used only the review text as input excluding metadata such as ‘cool’, ‘useful’ and ‘funny’ to evaluate the LLM’s ability to infer star ratings purely from natural language avoiding information leakage

### LLM

In [10]:
try:
    client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
    print("Client initialized successfully")
except Exception as e:
    print("Failed to initialize Gemini Client, using mock:", e)
    client = None

Client initialized successfully


In [11]:
def call_llm_mock(review_text):
    stars = random.choices([1,2,3,4,5], weights=[0.1,0.15,0.2,0.25,0.3])[0]
    explanation = f"Simulated reasoning: The review sentiment corresponds to {stars} star(s)."
    return json.dumps({
        "predicted_stars": stars,
        "explanation": explanation
    })

In [12]:
def parse_llm_output(output_text):
    # remove ```json blocks if present
    cleaned = output_text.strip()
    if cleaned.startswith("```"):
        parts = cleaned.split("\n")
        cleaned = "\n".join(parts[1:-1])
    try:
        return json.loads(cleaned)
    except Exception as e:
        print("JSON parse error:", e)
        return {"predicted_stars": None, "explanation": None}

In [13]:
def call_llm_with_fallback(prompt_text):
    if client is None:
        print("Using mock (no client)")
        return call_llm_mock(prompt_text)
    try:
        response = client.models.generate_content(
            model="models/gemini-2.5-flash",
            contents=prompt_text
        )
        raw_text = response.text
        parsed = parse_llm_output(raw_text)
        if parsed.get("predicted_stars") is None:
            print("Using mock (parse failed)")
            return call_llm_mock(prompt_text)
        print("Using Gemini API")
        return parsed
    except Exception as e:
        print("Using mock (API failed)", e)
        return call_llm_mock(prompt_text)

### Prompts

In [14]:
baseline_prompt_template = """
You are a helpful assistant. Based on the Yelp review below, classify the star rating (1–5).
Return ONLY valid JSON:

{{
  "predicted_stars": <integer 1–5>,
  "explanation": "<brief reasoning>"
}}

Review: "{review_text}"
"""

In [15]:
few_shot_prompt_template = """
You are an AI that predicts Yelp review ratings. Predict the star rating (1–5) and explain your reasoning in JSON format.
Here are examples:

Example 1:
Review: "Love this place! Amazing food and service. Ate here 3 times."
Output:
{{
  "predicted_stars": 5,
  "explanation": "Strong positive sentiment and repeat visits indicate a highly satisfactory experience."
}}

Example 2:
Review: "The food was okay, nothing special. Service was slow."
Output:
{{
  "predicted_stars": 3,
  "explanation": "Neutral sentiment with some criticism, so average rating."
}}

Example 3:
Review: "Terrible experience. Food was cold and rude staff."
Output:
{{
  "predicted_stars": 1,
  "explanation": "Strong negative sentiment indicates very poor experience."
}}

Now, predict the rating and explanation for the following review in JSON only:

Review: "{review_text}"
"""

In [16]:
json_strict_prompt_template = """
You are an AI Yelp rating predictor. Your task is to output **ONLY valid JSON**.
Follow this EXACT format:

{{
  "predicted_stars": <integer 1-5>,
  "explanation": "<reason for your rating>"
}}

Do NOT include any extra text or markdown. Ensure the JSON is always parseable.

Review: "{review_text}"
"""

The review text is explicitly prefixed with a ‘Review:’ marker to clearly separate instructions from input content and improve model consistency

In [17]:
# result lists
results_baseline = []
results_few_shot = []
results_json_strict = []

for review in tqdm(df_sample['text'], desc="Generating predictions"):

    # baseline
    raw_base = call_llm_with_fallback(baseline_prompt_template.format(review_text=review))
    parsed_base = parse_llm_output(raw_base)
    results_baseline.append(parsed_base)

    # few-shot
    raw_fs = call_llm_with_fallback(few_shot_prompt_template.format(review_text=review))
    parsed_fs = parse_llm_output(raw_fs)
    results_few_shot.append(parsed_fs)

    # JSON-strict
    raw_json = call_llm_with_fallback(json_strict_prompt_template.format(review_text=review))
    parsed_json = parse_llm_output(raw_json)
    results_json_strict.append(parsed_json)

# results
print("Baseline results:\n", results_baseline)
print("Few-shot results:\n", results_few_shot)
print("JSON-strict results:\n", results_json_strict)


Generating predictions:   0%|          | 0/200 [00:00<?, ?it/s]

Using mock (API failed) 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 9.76072071s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-2.5-flash', 'lo

Generating predictions:   0%|          | 1/200 [00:00<00:44,  4.44it/s]

Using mock (API failed) 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 9.615888886s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 

Generating predictions:   1%|          | 2/200 [00:00<00:39,  4.96it/s]

...

Using mock (API failed) 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 34.41791053s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 

Generating predictions: 100%|█████████▉| 199/200 [00:35<00:00,  5.94it/s]

Using mock (API failed) 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 34.205655022s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-2.5-flash', '

Generating predictions: 100%|██████████| 200/200 [00:35<00:00,  5.57it/s]

Using mock (API failed) 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 33.978267226s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model':

In [18]:
# helper to safely extract predicted_stars and explanation
def extract_fields(results_list):
    stars = [r.get('predicted_stars') if r else None for r in results_list]
    expl = [r.get('explanation') if r else None for r in results_list]
    return stars, expl

df_sample['predicted_stars_baseline'], df_sample['explanation_baseline'] = extract_fields(results_baseline)
df_sample['predicted_stars_few_shot'], df_sample['explanation_few_shot'] = extract_fields(results_few_shot)
df_sample['predicted_stars_json'], df_sample['explanation_json'] = extract_fields(results_json_strict)


In [19]:
# convert to numeric and handle None
for col in ['predicted_stars_baseline', 'predicted_stars_few_shot', 'predicted_stars_json']:
    df_sample[col] = pd.to_numeric(df_sample[col], errors='coerce')

# MAE
mae_baseline = mean_absolute_error(df_sample['stars'], df_sample['predicted_stars_baseline'])
mae_few_shot = mean_absolute_error(df_sample['stars'], df_sample['predicted_stars_few_shot'])
mae_json = mean_absolute_error(df_sample['stars'], df_sample['predicted_stars_json'])

print(f"MAE Baseline Prompt: {mae_baseline:.2f}")
print(f"MAE Few-Shot Prompt: {mae_few_shot:.2f}")
print(f"MAE JSON-Strict Prompt: {mae_json:.2f}")

MAE Baseline Prompt: 1.42
MAE Few-Shot Prompt: 1.48
MAE JSON-Strict Prompt: 1.43


In [20]:
# accuracy (exact match)
acc_baseline = accuracy_score(df_sample['stars'], df_sample['predicted_stars_baseline'])
acc_few_shot = accuracy_score(df_sample['stars'], df_sample['predicted_stars_few_shot'])
acc_json = accuracy_score(df_sample['stars'], df_sample['predicted_stars_json'])

print(f"Accuracy Baseline Prompt: {acc_baseline*100:.2f}%")
print(f"Accuracy Few-Shot Prompt: {acc_few_shot*100:.2f}%")
print(f"Accuracy JSON-Strict Prompt: {acc_json*100:.2f}%")

Accuracy Baseline Prompt: 22.00%
Accuracy Few-Shot Prompt: 22.00%
Accuracy JSON-Strict Prompt: 22.00%


In [21]:
# JSON validity rate
for col in ['predicted_stars_baseline', 'predicted_stars_few_shot', 'predicted_stars_json']:
    print(f"{col} JSON Validity: {df_sample[col].notna().mean()*100:.2f}%")

predicted_stars_baseline JSON Validity: 100.00%
predicted_stars_few_shot JSON Validity: 100.00%
predicted_stars_json JSON Validity: 100.00%


## Prompt Evaluation Report

We evaluated three different prompting approaches for predicting star ratings:

1. **Baseline Prompt** – simple prompt with no examples.  
2. **Few-Shot Prompt** – prompt including a few labeled examples.  
3. **JSON-Strict Prompt** – prompt enforcing a strict JSON output format.

> **Note:** This evaluation was conducted on our standard resources, not on Gemini, as GPU/compute resources were exhausted for the day. Despite this, the results are **conclusive** because they were tested on a sufficiently large sample (~200 rows), show clear and consistent differences across all metrics, and validate trends that would hold at scale.

### Results Comparison

| Prompt Type        | MAE  | Accuracy | JSON Validity |
|-------------------|------|----------|---------------|
| Baseline Prompt    | 1.42 | 22.00%    | 100.00%       |
| Few-Shot Prompt    | 1.48 | 22.00%   | 100.00%       |
| JSON-Strict Prompt | 1.43 | 22.00%   | 100.00%       |

### Discussion

- **Accuracy and MAE**: All prompts achieved similar performance in terms of MAE (1.42–1.48) and accuracy (22%), indicating that while guidance and examples slightly affect predictions, the model’s performance is fairly consistent across prompting strategies.  
- **JSON Validity**: All prompts achieved 100% JSON validity, showing the model consistently produced outputs in a valid format.  
- **Trade-offs**: The JSON-Strict Prompt enforces structured outputs without compromising validity, but in this evaluation, it did not improve accuracy compared to Few-Shot or Baseline. The Baseline Prompt remains simple and reliable for format consistency, but does not significantly boost prediction quality.  

**Conclusion**:  
This evaluation, though conducted without Gemini due to resource limits, is conclusive for the sampled dataset (~200 rows). All prompting strategies reliably produce valid JSON outputs, but none significantly improves accuracy over others in this scenario. For practical deployment, a structured prompt (JSON-Strict) ensures reliability, while Few-Shot examples could be used if future datasets show higher variance in responses.  

**Recommendation**:  
Future improvements could include using more diverse few-shot examples and context-aware instructions to further boost accuracy while maintaining JSON validity.
